# Load datasets

In [2979]:
# Import necessary libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn as sk
import geopandas as gpd
import json
import ast

from shapely.geometry.base import BaseGeometry
from shapely.geometry import MultiPoint
from shapely.geometry import Point
from shapely.geometry import shape

pd.set_option("display.max_columns", None)

# Merge files to have a mapping of ward, sub, and neighbourhood for each voting subdivision

In [2980]:
# In other words, given the ward and subdivision data in our voting dataset, return the neighbourhood
# we do this since most interesting potential predictors are aggregated at the neighbourhood-level

## 1. Load and project spatial data

In [2981]:
# Get geojson data

subdivisions = gpd.read_file(
    "ZZ_voting/voting-subdivisions-2022 - 4326.geojson"
).to_crs("EPSG:26917")

neighbourhoods = gpd.read_file(
    "ZZ_voting/Neighbourhoods - historical 140 - 4326.geojson"
).to_crs("EPSG:26917")

wards = gpd.read_file(
    "ZZ_voting/city-wards-data-4326.geojson"
).to_crs("EPSG:26917")

## 2. Assign each subdivision to the neighbourhood containing the largest share of its area

In [2982]:
# Calculates the area of each voting subdivision in squared meters (EPSG:26917)

subdivisions["subdivision_area"] = subdivisions.area

# get neighbourhoods and their geometry
hoods = neighbourhoods[
    ["AREA_SHORT_CODE", "AREA_NAME", "geometry"]
].rename(columns={
    "AREA_SHORT_CODE": "NEIGHBOURHOOD_ID",
    "AREA_NAME": "NEIGHBOURHOOD"
})

# Take subdivision polygons and intersect them with the neighbourhood polygons
# One subdivision can appear multiple times, with polygons reflect the 
# coordinates of overlap (e.g., 80% Neighbourhood 1, 20% neighbourhood for each AREA_ID 
# (subdivision unique key)
overlap = gpd.overlay(
    subdivisions[
        ["AREA_ID", "subdivision_area", "geometry"]
    ],
    hoods,
    how="intersection"
)

# Calculate physical area of each subdivision-neighbourhood intersection (m^2)
overlap["overlap_area"] = overlap.area

# Sort intersections from largest to smallest
# Keep first AREA_ID (subdivision unique key)
# We therefore link the voting subdivision to the neighbourhood that it overlaps with the most
sub_to_hood = (
    overlap
    .sort_values(
        "overlap_area",
        ascending=False
    )
    .drop_duplicates("AREA_ID")
    [[
        "AREA_ID",
        "NEIGHBOURHOOD_ID",
        "NEIGHBOURHOOD"
    ]]
)

sub_to_hood

c:\Users\owner\OneDrive\Desktop\Projects\00_DataAnalysis_andAI\.venv\Lib\site-packages\geopandas\tools\overlay.py:375: UserWarning: `keep_geom_type=True` in overlay resulted in 1173 dropped geometries of different geometry types than df1 has. Set `keep_geom_type=False` to retain all geometries
  result = _collection_extract(result, geom_type, keep_geom_type_warning)


,AREA_ID,NEIGHBOURHOOD_ID,NEIGHBOURHOOD
2678,2508404.0,131,Rouge (131)
1221,2507817.0,001,West Humber-Clairville (1)
2311,2508437.0,077,Waterfront Communities-The Island (77)
363,2508138.0,026,Downsview-Roding-CFB (26)
1854,2508484.0,014,Islington-City Centre West (14)
...,...,...,...
19,2508168.0,077,Waterfront Communities-The Island (77)
152,2508040.0,017,Mimico (includes Humber Bay Shores) (17)
1810,2508274.0,075,Church-Yonge Corridor (75)
428,2507624.0,018,New Toronto (18)


## 3. Assign each subdivision to its ward


In [2983]:
sub_points = subdivisions[
    ["AREA_ID", "AREA_SHORT_CODE", "geometry"]
].copy()

# Rename area short code to sub 
sub_points["SUB"] = (
    sub_points["AREA_SHORT_CODE"]
    .astype(int)
)

# Get a representative point within the subdivision using GeoPandas/Shaply
# algorithm. This is fine to use to link wards because subdivisions are entirely 
# contained with a ward
sub_points["geometry"] = (
    sub_points.representative_point()
)

wards["WARD"] = (
    wards["AREA_SHORT_CODE"]
    .astype(int)
)

# Spatially join subdivision points to wards, identified wards linked to subdivisions
# Then merge neighbourhood mapping from previous code
sub_mapping = (
    gpd.sjoin(
        sub_points,
        wards[["WARD", "geometry"]],
        how="left",
        predicate="within"
    )
    [["AREA_ID", "WARD", "SUB"]]
    .merge(
        sub_to_hood,
        on="AREA_ID",
        how="left"
    )
)

sub_mapping

,AREA_ID,WARD,SUB,NEIGHBOURHOOD_ID,NEIGHBOURHOOD
0,2507308.0,11,55,079,University (79)
1,2507307.0,11,30,098,Rosedale-Moore Park (98)
2,2507306.0,6,1,035,Westminster-Branson (35)
3,2507305.0,5,42,110,Keelesdale-Eglinton West (110)
4,2508170.0,6,44,026,Downsview-Roding-CFB (26)
...,...,...,...,...,...
1455,2508282.0,21,60,138,Eglinton East (138)
1456,2508281.0,13,7,076,Bay Street Corridor (76)
1457,2508280.0,13,1,076,Bay Street Corridor (76)
1458,2508279.0,24,34,137,Woburn (137)


## 4. Prepare voting data DV

In [2984]:
# create a function to read files with an ID suffix appended to the end of the field
# This id suffix is used to link each variable to a specific source file from the 
# MasterList data dictionary
from pathlib import Path

def read_with_id(file_path, file_id):
    file_path = Path(file_path)

    if file_path.suffix.lower() in {".xls", ".xlsx", ".xlsm", ".xlsb"}:
        df = pd.read_excel(file_path)
    else:
        df = pd.read_csv(file_path)

    return df.add_suffix(f"_{file_id}")

In [2985]:
# Dependent variable dataframe for analysis
# Note 2022 data is used since that was last year Toronto had 140 neighbourhoods before it was changed to 158
# We use neighbourhood 140 since some datasets are older
df_dv = read_with_id('ZZ_voting/2022-voter-statistics.xlsx', "df_dv")
df_dv_interim = df_dv[[
    "Ward_df_dv",
    "Sub_df_dv",
    "TotalEligibleElectors_df_dv",
    "NumberVoted_df_dv"
]]
df_dv_interim.to_csv("test.csv")

In [2986]:
voters = df_dv_interim.copy()

voters["WARD"] = voters["Ward_df_dv"].astype(int)
voters["SUB"] = voters["Sub_df_dv"].astype(int)

# Remove non-geographic polls
# 97 = mail-in
# 98/99 = ward-wide advance
voters = voters[
    ~voters["SUB"].isin([97, 98, 99])
].copy()

voters

,Ward_df_dv,Sub_df_dv,TotalEligibleElectors_df_dv,NumberVoted_df_dv,WARD,SUB
0,1,1,1567.0,275,1,1
1,1,2,1656.0,355,1,2
2,1,3,1266.0,232,1,3
3,1,5,2669.0,284,1,5
4,1,6,1837.0,268,1,6
...,...,...,...,...,...,...
1527,25,47,1291.0,187,25,47
1528,25,48,2375.0,474,25,48
1529,25,49,3123.0,523,25,49
1530,25,52,289.0,53,25,52


## 5. Attach neighbourhoods to voter data

In [2987]:
# Validated some addresses from the 2022-voter-statistics data source with:
# https://www.toronto.ca/city-government/data-research-maps/neighbourhoods-communities/neighbourhood-profiles/find-your-neighbourhood/

voters_neighbourhood = voters.merge(
    sub_mapping[
        [
            "WARD",
            "SUB",
            "NEIGHBOURHOOD_ID",
            "NEIGHBOURHOOD"
        ]
    ],
    on=["WARD", "SUB"],
    how="left"
)

voters_neighbourhood.to_csv("voters_neighbourhood.csv")
voters_neighbourhood

,Ward_df_dv,Sub_df_dv,TotalEligibleElectors_df_dv,NumberVoted_df_dv,WARD,SUB,NEIGHBOURHOOD_ID,NEIGHBOURHOOD
0,1,1,1567.0,275,1,1,001,West Humber-Clairville (1)
1,1,2,1656.0,355,1,2,002,Mount Olive-Silverstone-Jamestown (2)
2,1,3,1266.0,232,1,3,002,Mount Olive-Silverstone-Jamestown (2)
3,1,5,2669.0,284,1,5,002,Mount Olive-Silverstone-Jamestown (2)
4,1,6,1837.0,268,1,6,002,Mount Olive-Silverstone-Jamestown (2)
...,...,...,...,...,...,...,...,...
1455,25,47,1291.0,187,25,47,136,West Hill (136)
1456,25,48,2375.0,474,25,48,136,West Hill (136)
1457,25,49,3123.0,523,25,49,136,West Hill (136)
1458,25,52,289.0,53,25,52,136,West Hill (136)


## 6. Aggregate to neighbourhood

In [2988]:
# Aggregate voting subdivisions to the neighbourhood-level
voters_neighbourhood_final = (
    voters_neighbourhood
    .groupby(
        ["NEIGHBOURHOOD_ID", "NEIGHBOURHOOD"],
        as_index=False
    )
    .agg(
        NumberVoted_df_dv=("NumberVoted_df_dv", "sum"),
        TotalEligibleElectors_df_dv=("TotalEligibleElectors_df_dv", "sum")
    )
    .reset_index()
)

# Create PercentVoted DV that takes into account the population size
# of each voting subdivision within neighbourhoods
voters_neighbourhood_final["PercentVoted_df_dv"] = (
    voters_neighbourhood_final["NumberVoted_df_dv"]
    / voters_neighbourhood_final["TotalEligibleElectors_df_dv"]
)

voters_neighbourhood_final = voters_neighbourhood_final[[
    "NEIGHBOURHOOD_ID",
    "NEIGHBOURHOOD",
    "PercentVoted_df_dv"
]]

# Build individual IVs

In [2989]:
# create seperate dataframes for each file with an ID suffix
# This links the variable names with the specific source dataset
df1 = read_with_id('Community_Safety_Indicators/Community Safety Indicators Cleaned.csv', "df1")
df2 = read_with_id('Cultural_Hotspot_Points_of_Interest/points-of-interest - 4326_cleaned.csv', "df2")
df3 = read_with_id('HateCrimes/Hate Crimes Open Data.csv', "df3")
df4 = read_with_id('Homicides/Homicides - 4326.csv', "df4")
df5 = read_with_id('Housing_Rent_and_Other/AboriginalHouseholds2021_cleaned.csv', "df5")
df6 = read_with_id('Housing_Rent_and_Other/construction2020.csv', "df6")
df7 = read_with_id('Housing_Rent_and_Other/construction2021.csv', "df7")
df8 = read_with_id('Housing_Rent_and_Other/construction2022.csv', "df8")
df9 = read_with_id('Housing_Rent_and_Other/construction2023.csv', "df9")
df10 = read_with_id('Housing_Rent_and_Other/construction2024.csv', "df10")
df11 = read_with_id('Housing_Rent_and_Other/construction2025.csv', "df11")
df12 = read_with_id('Housing_Rent_and_Other/HouseholdChildren_2021_cleaned.csv', "df12")
df13 = read_with_id('Housing_Rent_and_Other/HouseholdIncome2021a_cleaned.csv', "df13")
df14 = read_with_id('Housing_Rent_and_Other/HouseholdIncome2021b.csv', "df14")
df15 = read_with_id('Housing_Rent_and_Other/HouseholdSize2021_cleaned.csv', "df15")
df16 = read_with_id('Housing_Rent_and_Other/HouseholdType2021_cleaned.csv', "df16")
df17 = read_with_id('Housing_Rent_and_Other/HouseholdwithSeniors2021.csv', "df17")
df23 = read_with_id('Housing_Rent_and_Other/housing2025_cleaned.csv', "df23")
df24 = read_with_id('Housing_Rent_and_Other/Immigranthouseholds2021_cleaned.csv', "df24")
df25 = read_with_id('Mental_Health_Act/mental-health-apprehensions.csv', "df25")
df26 = read_with_id('Neighbourhood_Crime_Rates/neighbourhood-crime-rates-4326_cleaned.csv', "df26")
df27 = read_with_id('Neighbourhood_Improvement_Areas/Neighbourhood Improvement Areas - 4326_cleaned.csv', "df27")
df28 = read_with_id('Neighbourhood_Profiles/neighbourhood-profiles-2001-140-model.csv', "df28")
df29 = read_with_id('Neighbourhood_Profiles/neighbourhood-profiles-2006-140-model.csv', "df29")
df30 = read_with_id('Neighbourhood_Profiles/neighbourhood-profiles-2011-140-model.csv', "df30")
df31 = read_with_id('Neighbourhood_Profiles/neighbourhood-profiles-2016-140-model.csv', "df31")
df32 = read_with_id('Neighbourhood_Profiles/neighbourhood-profiles-2021-158-model_cleaned.xlsx', "df32")
df33 = read_with_id('Neighbourhoods/Neighbourhoods - 4326.csv', "df33")
df34 = read_with_id('Persons_in_Crisis/Persons in Crisis - Calls for Service Attended.csv', "df34")
df35 = read_with_id('Police_Arrested_Charged/Arrested and Charged Persons_cleaned.csv', "df35")
df36 = read_with_id('Police_Calls_for_Service/Calls for Service Attended.csv', "df36")
df37 = read_with_id('Police_Tickets_Issued/Tickets Issued.csv', "df37")
df38 = read_with_id('Shootings_Firearm_Discharges/shootings-firearm-discharges - 4326.csv', "df38")
df39 = read_with_id('Social_Housing_Unit_Density/social-housing-unit-density-by-neighbourhoods.xlsx', "df39")
df40 = read_with_id('Theft_from_Motor_Vehicle/theft-from-motor-vehicle - 4326.csv', "df40")
df41 = read_with_id('Wellbeing_Toronto_Civics_Equity/wellbeing-toronto-civics-equity-indicators_cleaned.xlsx', "df41")
df42 = read_with_id('Wellbeing_Toronto_Demographics/wellbeing-toronto-population-total-2011-2016-and-age-groups-2016.xlsx', "df42")
df43 = read_with_id('Wellbeing_Toronto_Demographics_NHS/demographics-nhs-indicators-2011.xlsx', "df43")
df44 = read_with_id('Wellbeing_Toronto_Demographics_NHS/education-nhs-indicators-2011.xlsx', "df44")
df45 = read_with_id('Wellbeing_Toronto_Economics/wellbeing-toronto-economics_cleaned.xlsx', "df45")
df46 = read_with_id('Wellbeing_Toronto_Education/wellbeing-toronto-education_cleaned.xlsx', "df46")
df47 = read_with_id('Wellbeing_Toronto_Environment/wellbeing-toronto-environment_cleaned.xlsx', "df47")
df48 = read_with_id('Wellbeing_Toronto_Health/wellbeing-toronto-health_cleaned.xlsx', "df48")
df49 = read_with_id('Wellbeing_Toronto_Housing/wellbeing-toronto-housing_cleaned.xlsx', "df49")
df50 = read_with_id('Wellbeing_Toronto_Safety/wellbeing-toronto-safety.xlsx', "df50")
df51 = read_with_id('ZZ_OntarioHealthProfiles/1_ACSC-Hosp_202123_TOR_v1.0_cleaned.xlsx', "df51")
df52 = read_with_id('ZZ_OntarioHealthProfiles/1_AHD_2pCC_FY2023_TOR_v1.0_cleaned.xlsx', "df52")
df53 = read_with_id('ZZ_OntarioHealthProfiles/1_AHD_Asthma_FY2022_TOR_v1.2_cleaned.xlsx', "df53")
df54 = read_with_id('ZZ_OntarioHealthProfiles/1_AHD_COPD_FY2023_TOR_d2023_cleaned.xlsx', "df54")
df55 = read_with_id('ZZ_OntarioHealthProfiles/1_AHD_Diabetes_FY2022_TOR_d2023_cleaned.xlsx', "df55")
df56 = read_with_id('ZZ_OntarioHealthProfiles/1_AHD_HBP_FY2022_TOR_d2023_cleaned.xlsx', "df56")
df57 = read_with_id('ZZ_OntarioHealthProfiles/1_AHD_MHV_CY2023_TOR_v1.0_cleaned.xlsx', "df57")
df58 = read_with_id('ZZ_OntarioHealthProfiles/1_ALL-Hosp_202123_TOR_v1.0_cleaned.xlsx', "df58")
df59 = read_with_id('ZZ_OntarioHealthProfiles/1_AnnualBirthRate_CY202022_TOR_v1.0_cleaned.xlsx', "df59")
df60 = read_with_id('ZZ_OntarioHealthProfiles/1_BirthsbyMCOB_CY201921_TOR_v1.0_cleaned.xlsx', "df60")
df61 = read_with_id('ZZ_OntarioHealthProfiles/1_ChildrenAndYouth_neighb_2015_LHIN_7_cleaned_asthma_youth.xls', "df61")
df62 = read_with_id('ZZ_OntarioHealthProfiles/1_ChildrenAndYouth_neighb_2015_LHIN_7_cleaned_injuries_ED.xls', "df62")
df63 = read_with_id('ZZ_OntarioHealthProfiles/1_ChildrenAndYouth_neighb_2015_LHIN_7_cleaned_mentalhealth_ED.xls', "df63")
df64 = read_with_id('ZZ_OntarioHealthProfiles/AHD_CHF_202223_TOR_v2.0_cleaned.xlsx', "df64")
df65 = read_with_id('ZZ_OntarioHealthProfiles/EDV_All_HU_LU_202325_TOR_v1.0_cleaned.xlsx', "df65")
df66 = read_with_id('ZZ_OntarioHealthProfiles/EDV_MHA_202325_TOR_v1.0_cleaned.xlsx', "df66")
df67 = read_with_id('ZZ_OntarioHealthProfiles/MHA-Hosp_202123_TOR_v1.0_cleaned.xlsx', "df67")
df68 = read_with_id('ZZ_OntarioHealthProfiles/PrematureMortality_202123_TOR_v1.0_cleaned.xlsx', "df68")
df69 = read_with_id('ZZ_OntarioHealthProfiles/SH_Chlam_Gon_201822_TOR_v1.0_cleaned_Chlamydia.xlsx', "df69")
df70 = read_with_id('ZZZ_Affordable_Housing/Affordable Rental Housing Pipeline - 4326_cleaned.csv', "df70")
df71 = read_with_id('ZZZ_Bridges/bridges-structure - 4326.csv', "df71")
df72 = read_with_id('ZZZ_CafeTO_Locations/Cafe TO Locations - 4326.csv', "df72")
df73 = read_with_id('ZZZ_Clothing_Drop_Box/Clothing Drop-Box Locations - 4326.csv', "df73")
df74 = read_with_id('ZZZ_Drinking_Fountains/Parks Drinking Fountains - 4326.csv', "df74")
df75 = read_with_id('ZZZ_Fire_Hydrants/Fire Hydrants Data - 4326.csv', "df75")
df76 = read_with_id('ZZZ_Food_Inspections/Dinesafe_cleaned.csv', "df76") 
df77 = read_with_id('ZZZ_Heat_Relief_Network/Air Conditioned and Cool Spaces - 4326.csv', "df77")
df78 = read_with_id('ZZZ_Motor_Vehicle_Collisions/Motor Vehicle Collisions with KSI Data - 4326.csv', "df78")
df79 = read_with_id('ZZZ_Park_Washrooms/Washroom Facilities - 4326.csv', "df79")
df80 = read_with_id('ZZZ_Parks_and_Rec/Parks and Recreation Facilities - 4326.csv', "df80")
df81 = read_with_id('ZZZ_Places_of_Worship/Places of Worship - 4326.csv', "df81")
df82 = read_with_id('ZZZ_PSS_License/bodysafe - 4326.csv', "df82")
df83 = read_with_id('ZZZ_Public_Art/Public Art - 4326.csv', "df83")
df84 = read_with_id('ZZZ_Real_Estate_Assets/building-asset-inventory - 4326.csv', "df84")
df85 = read_with_id('ZZZ_Real_Estate_Assets/land-asset-inventory - 4326.csv', "df85")
df86 = read_with_id('ZZZ_School_Safety_Zone/School Safety Zone Watch Your Speed Program - Locations - 4326.csv', "df86")
df87 = read_with_id('ZZZ_School_Safety_Zone/Stationary Sign locations - 4326.csv', "df87")
df88 = read_with_id('ZZZ_Schools/School locations-all types data - 4326.csv', "df88")
df89 = read_with_id('ZZZ_Street_Benches/Street furniture-Bench data - 4326.csv', "df89")
df90 = read_with_id('ZZZ_Tennis_Courts/Tennis Courts Facilities - 4326.csv', "df90")
df91 = read_with_id('ZZZ_Toronto_Property_Sale/Toronto Signature Sites - 4326.csv', "df91")
df92 = read_with_id('ZZZ_Traffic_Volumes/tmc_most_recent_summary_data_cleaned.csv', "df92")
df93 = read_with_id('ZZZ_Trees/Street Tree Data - 4326.csv', "df93")


C:\Users\owner\AppData\Local\Temp\ipykernel_20848\3777794801.py:12: DtypeWarning: Columns (7: SUFFIX) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path)


In [2990]:
# create linkage table, where each hood158 is associated with hood140, use df25 to identify full neighbourhood mapping
linkage = df25[["HOOD_158_df25",
    "NEIGHBOURHOOD_158_df25",
    "HOOD_140_df25",
    "NEIGHBOURHOOD_140_df25"]]

linkage = linkage.rename(columns={
    "HOOD_158_df25":"HOOD_158",
    "NEIGHBOURHOOD_158_df25":"NEIGHBOURHOOD_158",
    "HOOD_140_df25":"HOOD_140",
    "NEIGHBOURHOOD_140_df25":"NEIGHBOURHOOD_140"
})

# Remove duplicates to give a cleaned 158 -> 140 mapping
linkage = linkage.drop_duplicates().reset_index(drop=True).sort_values("HOOD_158")

# Drop NSA
linkage = linkage[linkage["HOOD_158"].astype(str).str.strip() != "NSA"].copy()

# Drop () so that we can match by neighbourhood name if needed later
linkage[["NEIGHBOURHOOD_158_cleaned", "NEIGHBOURHOOD_140_cleaned"]] = (
    linkage[["NEIGHBOURHOOD_158", "NEIGHBOURHOOD_140"]]
    .apply(lambda col: col.str.replace(r"\s*\(\d+\)$", "", regex=True))
)

linkage_table = linkage
linkage_table.to_csv("cleaned_hood_mapping.csv")

# Convert neighbourhood 158 to int for later joins
linkage_table["HOOD_158"] = pd.to_numeric(
    linkage_table["HOOD_158"],
    errors="coerce"
)

linkage_table = linkage_table.dropna(
    subset=["HOOD_158"]
)

linkage_table["HOOD_158"] = (
    linkage_table["HOOD_158"].astype(int)
)

linkage_table


PermissionError: [Errno 13] Permission denied: 'cleaned_hood_mapping.csv'

In [ ]:
# preprocess df1 (Community Safety Indicators), converting row-wise dataframe to the neighbourhood-level
# Note OFFENCE_df1 is in raw counts, we derive rate per 100 later this notebook
df1_final = (
    df1.groupby("HOOD_140_df1")["OFFENCE_df1"]
    .count()
    .reset_index()
)

In [ ]:
# Help preprocess shapely geometries
def convert_geometry(x):
    # Already a Shapely geometry
    if isinstance(x, BaseGeometry):
        return x

    # JSON stored as a string
    if isinstance(x, str):
        return shape(json.loads(x))

    # Already parsed JSON/dictionary
    if isinstance(x, dict):
        return shape(x)

    return None

In [ ]:
# Cultural hotspots; first preprocess
df2_geo = df2.copy()

df2_geo["geometry_df2"] = (
    df2_geo["geometry_df2"]
    .apply(convert_geometry)
)

# Tell GeoPandas that the original hotspot coordinates are lon/lat
df2_geo = gpd.GeoDataFrame(
    df2_geo,
    geometry="geometry_df2",
    crs="EPSG:4326"
)

# Convert cultural hotspots to the same CRS as neighbourhoods
df2_geo = df2_geo.to_crs(neighbourhoods.crs)

# Identify neighbourhood for each cultural hotspot
df2_neighbourhood = gpd.sjoin(
    df2_geo,
    neighbourhoods[
        ["AREA_SHORT_CODE", "AREA_NAME", "geometry"]
    ],
    how="left",
    predicate="within"
).rename(columns={
    "AREA_SHORT_CODE": "HOOD_140_df2",
    "EntryType_df2": "CulturalHotspots_df2"
})

# Count cultural hotspots within each neighbourhood
hotspot_counts = (
    df2_neighbourhood
    .groupby("HOOD_140_df2")
    .size()
    .reset_index(name="CulturalHotspots_df2")
)

# Start with ALL neighbourhoods
df2_final = (
    neighbourhoods[
        ["AREA_SHORT_CODE"]
    ]
    .drop_duplicates()
    .rename(columns={
        "AREA_SHORT_CODE": "HOOD_140_df2"
    })
    .merge(
        hotspot_counts,
        on="HOOD_140_df2",
        how="left"
    )
)

# Neighbourhoods with no cultural hotspots get 0
df2_final["CulturalHotspots_df2"] = (
    df2_final["CulturalHotspots_df2"]
    .fillna(0)
    .astype(int)
)

In [ ]:
# Complete list of all 140 neighbourhood IDs
all_hoods_140 = (
    linkage_table[["HOOD_140"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

In [ ]:
# Function for counting variables
def create_count_df(
    df,
    hood_col,
    count_col,
    output_col
):
    
    # Remove NSA / invalid neighbourhood assignments
    temp = df[
        df[hood_col].notna() &
        (df[hood_col] != "NSA")
    ]

    # Count observations within each neighbourhood
    counts = (
        temp
        .groupby(hood_col)
        [count_col]
        .count()
        .reset_index()
        .rename(columns={
            count_col: output_col
        })
    )

    # Start with all 140 neighbourhoods
    final = (
        all_hoods_140
        .rename(columns={
            "HOOD_140": hood_col
        })
        .merge(
            counts,
            on=hood_col,
            how="left"
        )
    )

    # Neighbourhoods with no observations get 0
    final[output_col] = (
        final[output_col]
        .fillna(0)
        .astype(int)
    )

    return final

In [ ]:
# Hate crimes
df3_final = create_count_df(
    df=df3,
    hood_col="HOOD_140_df3",
    count_col="PRIMARY_OFFENCE_df3",
    output_col="HateCrimes_df3"
)

In [ ]:
# Homicides
df4_final = create_count_df(
    df=df4,
    hood_col="HOOD_140_df4",
    count_col="HOMICIDE_TYPE_df4",
    output_col="Homicides_df4"
)

In [ ]:
# Aboriginal Households, a lot of missing neighbourhoods due to different mapping, 
# these will be NaN when joined later
# 2021 data, so use neighbourhood 140 
df5_interim = df5.merge(
    linkage_table,
    left_on = "Neighbourhood_df5",
    right_on = "NEIGHBOURHOOD_140_cleaned",
    how = "left"
)
df5_final = df5_interim.groupby("HOOD_140")["AboriginalHouseholdRate_df5"].mean().reset_index().rename(columns={"HOOD_140":"HOOD_140_df5"})

In [ ]:
# Proportion of households with Children, a lot of missing neighbourhoods due to different mapping, 
# these will be NaN when joined later
# 2021 data, so use neighbourhood 140 
df12_interim = df12.merge(
    linkage_table,
    left_on = "Neighbourhood_df12",
    right_on = "NEIGHBOURHOOD_140_cleaned",
    how = "left"
)
df12_final = df12_interim.groupby("HOOD_140")["ChildrenRate_df12"].mean().reset_index().rename(columns={"HOOD_140":"HOOD_140_df12"})

In [ ]:
# Median household income 
# 2021 data so use neighbourhood 140
df13_interim = df13.merge(
    linkage_table,
    left_on = "Neighbourhood_df13",
    right_on = "NEIGHBOURHOOD_140_cleaned",
    how = "left"
)
df13_final = df13_interim.groupby("HOOD_140")["MedianHouseholdIncomeBeforeTaxes_df13"].mean().reset_index().rename(columns={"HOOD_140":"HOOD_140_df13"})


In [ ]:
# Proportion of households with 5 or more individuals
# 2021 data so use neighbourhood 140
df15_interim = df15.merge(
    linkage_table,
    left_on = "Neighbourhood_df15",
    right_on = "NEIGHBOURHOOD_140_cleaned",
    how = "left"
)
df15_final = df15_interim.groupby("HOOD_140")["LargeHouseholdRate_df15"].mean().reset_index().rename(columns={"HOOD_140":"HOOD_140_df15"})

In [ ]:
# Proportion of households with couples with children
# 2021 data
df16_interim = df16.merge(
    linkage_table,
    left_on = "Neighbourhood_df16",
    right_on = "NEIGHBOURHOOD_140_cleaned",
    how = "left"
)
df16_final = df16_interim.groupby("HOOD_140")["CoupleWithChildrenRate_df16"].mean().reset_index().rename(columns={"HOOD_140":"HOOD_140_df16"})

In [ ]:
# Avg Home Price
# 2025 data, but more rows matched using 140 mapping
df23_interim = df23[df23["AvgRent_df23"] != "**"]
df23_interim["AvgRent_df23"] = df23_interim["AvgRent_df23"].astype(int)
df23_interim = df23_interim.merge(
    linkage_table,
    left_on = "Neighbourhood_df23",
    right_on = "NEIGHBOURHOOD_140_cleaned",
    how = "left"
)
df23_final = df23_interim.groupby("HOOD_140")["AvgRent_df23"].mean().reset_index().rename(columns={"HOOD_140":"HOOD_140_df23"})

In [ ]:
# Proportion of individuals that are recent immigrants landed between 
# 2016-2021, and total immigrant proportion 2021 data
df24_interim = df24.merge(
    linkage_table,
    left_on = "Neighbourhood_df24",
    right_on = "NEIGHBOURHOOD_140_cleaned",
    how = "left"
)
df24_final = df24_interim.groupby("HOOD_140")[["RecentImmigrantRate_df24","OverallImmigrantRate_df24"]].mean().reset_index().rename(columns={"HOOD_140":"HOOD_140_df24"})

In [ ]:
# Mental Health Act Apprehensions 
df25_final = create_count_df(
    df=df25,
    hood_col="HOOD_140_df25",
    count_col="APPREHENSION_TYPE_df25",
    output_col="MentalHealthApprehensions_df25"
)

In [ ]:
# Crime Rate
# 2025 data
df26_merge = df26.merge(
    linkage_table,
    left_on = "HOOD_ID_df26",
    right_on = "HOOD_158",
    how = "left"
).rename(columns={"HOOD_140":"HOOD_140_df26"})
df26_final = df26_merge.groupby("HOOD_140_df26")["CrimeRatePer100_df26"].mean().reset_index()

In [ ]:
# Neighbourhood Improvement Area
# Many missing neighbourhoods
df27_interim = linkage_table.merge(
    df27,
    left_on="HOOD_158", 
    right_on="AREA_SHORT_CODE_df27",
    how="left"
)
df27_interim["NeighbourhoodImprovementPlan_df27"] = df27_interim["AREA_SHORT_CODE_df27"].notna().astype(int)
df27_final = df27_interim.groupby("HOOD_140")["NeighbourhoodImprovementPlan_df27"]\
    .sum()\
    .reset_index()\
    .rename(columns={"HOOD_140":"HOOD_140_df27"}) # sum to identify neighbourhood 140s that have two neighbourhood 158
df27_final["NeighbourhoodImprovementPlan_df27"] = df27_final["NeighbourhoodImprovementPlan_df27"].replace(2, 1) # replace the summed with 1 indicating NIA

In [ ]:
# Neighbourhood profiles
# preprocess, selecting a set of interesting variables
# This dataset has complete neighbourhood 158 data
df32_merge = df32.merge(
    linkage_table,
    left_on = "Neighbourhood_df32",
    right_on = "HOOD_158",
    how = "left"
).rename(columns={"HOOD_140":"HOOD_140_df32"})

df32_merge.columns = df32_merge.columns.str.strip()

cols = [
    "Total Population_df32",
    "SeniorRate_df32",
    "Married or living common-law rate_df32", # Higher means more marriage/common law
    "House_rate_df32", # Heigher means higher proportion that lives in a house, as opposed to an apartment
    "Average household size_df32",
    "Median total income of household in 2020 ($)_df32",
    "Occupied private dwellings by period of construction, 2016 to 2021 rate_df32", # Higher means living in a more recently constructed home (from 2016-2021)
    "Mother Tongue English or French_df32", # Higher means more mother tongues in english or french as opposed to another language
    "Indigenous Rate_df32", # Proportion of indigenous peoples
    "Canadian Citizen Rate_df32", # Higher means more canadian citizens
    "Non-immigrant Rate_df32", # Higher means more non-immigrants
    "Unemployment rate_df32",
    "Advanced Degree Rate_df32", # Proportion of degree holders with a masters or phd
    "Long commute distance rate_df32" # Proportion of commutes 60 minutes and longer
]

# Total population should be SUMMED when multiple 158-neighbourhoods
# map to the same historical 140-neighbourhood.
# All other variables are averaged.

population_col = "Total Population_df32"

mean_cols = [
    col for col in cols
    if col != population_col
]

aggregation_rules = {
    population_col: lambda x: x.sum(min_count=1),
    **{col: "mean" for col in mean_cols}
}

df32_final = (
    df32_merge
    .groupby(
        "HOOD_140_df32",
        as_index=False
    )
    .agg(aggregation_rules)
)

In [ ]:
# Arrested and Charged Persons
df35_interim = df35[df35["CATEGORY_df35"] != "Total Arrests"] # dataset includes total arrests summed as individal rows, so remove
df35_interim = df35_interim[df35_interim["HOOD_158_df35"] != "NSA"] 
df35_interim["HOOD_158_df35"] = df35_interim["HOOD_158_df35"].astype(int)
df35_interim = df35_interim.merge(linkage_table, left_on="HOOD_158_df35", right_on="HOOD_158", how="left")
df35_interim = df35_interim.groupby(["HOOD_140","AGE_GROUP_df35"])["ARREST_COUNT_df35"].sum().reset_index()

# Get youth arrested rate. This is a ratio - youth crime rate relative to all crimes committed 
df35_youth = df35_interim[df35_interim["AGE_GROUP_df35"]=="Youth"]
df35_adult = df35_interim[df35_interim["AGE_GROUP_df35"]=="Adult"]
df35_join = df35_youth.merge(df35_adult, on="HOOD_140", how="inner")
df35_join["YouthCrimeRate_df35"] = df35_join["ARREST_COUNT_df35_x"]/(df35_join["ARREST_COUNT_df35_x"]+df35_join["ARREST_COUNT_df35_y"])
df35_final = df35_join[["HOOD_140","YouthCrimeRate_df35"]].rename(columns={"HOOD_140":"HOOD_140_df35"})

In [ ]:
# Police Calls for Service
# Convert to rate later
df36_interim = df36[df36["HOOD_158_df36"] != 'NSA']
df36_interim["HOOD_158_df36"] = df36_interim["HOOD_158_df36"].astype(int) 
df36_interim = df36_interim.merge(linkage_table,left_on="HOOD_158_df36", right_on="HOOD_158",how="left")
df36_final = df36_interim.groupby("HOOD_140")["EVENT_COUNT_df36"].sum().reset_index().rename(columns={"HOOD_140":"HOOD_140_df36","EVENT_COUNT_df36":"PoliceCallsAttended_df36"})

In [ ]:
# Police Tickets Issued
# convert to rate later
df37_interim = df37[df37["HOOD_158_df37"] != 'NSA']
df37_interim["HOOD_158_df37"] = df37_interim["HOOD_158_df37"].astype(int)
df37_interim = df37_interim.merge(
    linkage_table,
    left_on="HOOD_158_df37",
    right_on="HOOD_158",
    how="left"
)
df37_final = df37_interim.groupby("HOOD_140")["TICKET_COUNT_df37"]\
    .sum()\
    .reset_index()\
    .rename(columns={"HOOD_140":"HOOD_140_df37"})

In [ ]:
# Fire arm discharges
df38_interim = df38[df38["HOOD_158_df38"] != 'NSA']
df38_final = df38_interim.groupby("HOOD_140_df38")["EVENT_TYPE_df38"]\
    .count()\
    .reset_index()\
    .rename(columns={"HOOD_140":"HOOD_140_df38","EVENT_TYPE_df38":"FirearmDisarges_df38"})

In [ ]:
# Social housing units
# Convert to rate later
# Assume missing neighbourhoods have 0 social housing units

df39_interim = df39.rename(columns={
    "Neighbourhood_df39": "HOOD_140_df39",
    "Units_df39": "SocialHousingUnits_df39"
})

# Standardize neighbourhood IDs to 3-digit strings
df39_interim["HOOD_140_df39"] = (
    df39_interim["HOOD_140_df39"]
    .astype(int)
    .astype(str)
    .str.zfill(3)
)

# Sum units in case there is more than one row per neighbourhood
# divide by sum of population later
df39_interim = (
    df39_interim
    .groupby("HOOD_140_df39")["SocialHousingUnits_df39"]
    .sum()
    .reset_index()
)

# Complete list of all 140 neighbourhoods
all_hoods_140 = (
    linkage_table[["HOOD_140"]]
    .drop_duplicates()
    .rename(columns={
        "HOOD_140": "HOOD_140_df39"
    })
)

# Standardize IDs
all_hoods_140["HOOD_140_df39"] = (
    all_hoods_140["HOOD_140_df39"]
    .astype(str)
    .str.zfill(3)
)

# Keep all 140 neighbourhoods
df39_final = (
    all_hoods_140
    .merge(
        df39_interim,
        on="HOOD_140_df39",
        how="left"
    )
)

# Missing neighbourhoods = 0 social housing units
df39_final["SocialHousingUnits_df39"] = (
    df39_final["SocialHousingUnits_df39"]
    .fillna(0)
)

In [ ]:
def create_158_count_df(
    df,
    hood_158_col,
    count_col,
    output_hood_col,
    output_count_col,
    linkage_table
):

    # Remove records without a valid neighbourhood
    temp = df[
        df[hood_158_col].notna() &
        (df[hood_158_col] != "NSA")
    ].copy()

    # Convert neighbourhood ID to integer
    temp[hood_158_col] = (
        temp[hood_158_col]
        .astype(int)
    )

    # Map 158-neighbourhood IDs to 140-neighbourhood IDs
    temp = temp.merge(
        linkage_table[
            ["HOOD_158", "HOOD_140"]
        ],
        left_on=hood_158_col,
        right_on="HOOD_158",
        how="left"
    )

    # Count events within each 140 neighbourhood
    counts = (
        temp
        .groupby("HOOD_140")[count_col]
        .count()
        .reset_index()
        .rename(columns={
            "HOOD_140": output_hood_col,
            count_col: output_count_col
        })
    )

    # Complete list of all 140 neighbourhoods
    all_hoods_140 = (
        linkage_table[
            ["HOOD_140"]
        ]
        .drop_duplicates()
        .rename(columns={
            "HOOD_140": output_hood_col
        })
    )

    # Merge counts onto all neighbourhoods
    final = (
        all_hoods_140
        .merge(
            counts,
            on=output_hood_col,
            how="left"
        )
    )

    # Neighbourhoods with no events get 0
    final[output_count_col] = (
        final[output_count_col]
        .fillna(0)
        .astype(int)
    )

    return final

In [ ]:
# Theft from motor vehicle
# Convert to rate later 
df40_final = create_158_count_df(
    df=df40,
    hood_158_col="HOOD_158_df40",
    count_col="OFFENCE_df40",
    output_hood_col="HOOD_140_df40",
    output_count_col="TheftFromMotorVehicle_df40",
    linkage_table=linkage_table
)

In [ ]:
# Equity indicators, Equity score is a composite score of many features we already have in this dataset, 
# like mental health, premature mortality, unemployment, low income, etc
# convert some to rates later
df41["Neighbourhood Id_df41"] = (
    df41["Neighbourhood Id_df41"]
    .astype(int)
    .astype(str)
    .str.zfill(3) # added leading 0s for consistency
)
df41_final = df41.rename(columns={"Neighbourhood Id_df41":"HOOD_140_df41"}).drop(columns="Neighbourhood_df41")

In [ ]:
# Economic data
df45["Neighbourhood Id_df45"] = (
    df45["Neighbourhood Id_df45"]
    .astype(int)
    .astype(str)
    .str.zfill(3) # added leading 0s for consistency
)
df45_final = df45.rename(columns={"Neighbourhood Id_df45":"HOOD_140_df45"}).drop(columns="Neighbourhood_df45")

In [ ]:
#  The EDI is a population-based measure of children’s readiness to learn in school and is administered by teachers at the Senior Kindergarten level
# 1 = low, 50 = medium, 100 = high, remap to 1, 2, 3
df46 = df46.rename(columns={"Neighbourhood Id_df46":"HOOD_140_df46"})
df46["HOOD_140_df46"] = (
    df46["HOOD_140_df46"]
    .astype(int)
    .astype(str)
    .str.zfill(3) # added leading 0s for consistency
)
df46["Early Development Instrument_df46"] = df46["Early Development Instrument_df46"].replace({1:1,50:2,100:3})
df46_final = df46.drop(columns="Neighbourhood_df46")

In [ ]:
# Environment
# Neighbourhoods vary in size, so tree cover for example is greater in larger neighbourhoods
# Divide by population later to identify e.g., amount of trees per person
df47 = df47.rename(columns={"Neighbourhood Id_df47":"HOOD_140_df47"})
df47["HOOD_140_df47"] = (
    df47["HOOD_140_df47"]
    .astype(int)
    .astype(str)
    .str.zfill(3) # added leading 0s for consistency
)
df47_final = df47.drop(columns="Neighbourhood_df47")

In [ ]:
# Health
df48 = df48.rename(columns={"Neighbourhood Id_df48":"HOOD_140_df48"})
df48["HOOD_140_df48"] = (
    df48["HOOD_140_df48"]
    .astype(int)
    .astype(str)
    .str.zfill(3) # added leading 0s for consistency
).copy()
df48_final = df48.drop(columns="Neighbourhood_df48")

In [ ]:
# Housing
df49 = df49.rename(columns={"Neighbourhood Id_df49":"HOOD_140_df49"})
df49["HOOD_140_df49"] = (
    df49["HOOD_140_df49"]
    .astype(int)
    .astype(str)
    .str.zfill(3) # added leading 0s for consistency
).copy()
df49_final = df49.drop(columns=["Neighbourhood_df49","Home Prices_df49"]) # home prices are duplicated from df45

In [ ]:
# Hospital Admissions: Rate (/100,000) of Ambulatory Care Sensitive Condition Hospitalizations between April 1, 2021 and March 31, 2023 
# for Neighbourhoods in the City of Toronto
df51_merge = df51.merge(
    linkage_table,
    left_on = "Neighbourhood_df51",
    right_on = "HOOD_158",
    how = "left"
).rename(columns={"HOOD_140":"HOOD_140_df51"})
df51_final = df51_merge.groupby("HOOD_140_df51")["Rate_HospitalAdmissions_df51"].mean().reset_index()

In [ ]:
# Adult Health and Disease: Prevalence (/100) of 2 or more Chronic Conditions on April 1st, 2023ª for Neighbourhoods in the City of Toronto
df52_merge = df52.merge(
    linkage_table,
    left_on = "Neighbourhood_df52",
    right_on = "HOOD_158",
    how = "left"
).rename(columns={"HOOD_140":"HOOD_140_df52"})
df52_final = df52_merge.groupby("HOOD_140_df52")["Prevalence_2+_ChronicConditions_df52"].mean().reset_index()

In [ ]:
# Prevalence (/100) of Asthma on April 1st, 2022 for Neighbourhoods in the City of Toronto
df53_merge = df53.merge(
    linkage_table,
    left_on = "Neighbourhood_df53",
    right_on = "HOOD_158",
    how = "left"
).rename(columns={"HOOD_140":"HOOD_140_df53"})
df53_final = df53_merge.groupby("HOOD_140_df53")["Prevalence_Asthma_df53"].mean().reset_index()

In [ ]:
# Adult Health and Disease: Prevalence (/100) of Chronic Obstructive Pulmonary Disease (COPD) on April 1st, 2023ª for Neighbourhoods in the City of Toronto
df54_merge = df54.merge(
    linkage_table,
    left_on = "Neighbourhood_df54",
    right_on = "HOOD_158",
    how = "left"
).rename(columns={"HOOD_140":"HOOD_140_df54"})
df54_final = df54_merge.groupby("HOOD_140_df54")["Pravalence_COPD_df54"].mean().reset_index()

In [ ]:
# Adult Health and Disease: Prevalence (/100) of Diabetes on April 1st, 2022 for Neighbourhoods in the City of Toronto
df55_merge = df55.merge(
    linkage_table,
    left_on = "Neighbourhood_df55",
    right_on = "HOOD_158",
    how = "left"
).rename(columns={"HOOD_140":"HOOD_140_df55"})
df55_final = df55_merge.groupby("HOOD_140_df55")["Pravelence_Diabetes_df55"].mean().reset_index()

In [ ]:
# Adult Health and Disease: Prevalence (/100) of High Blood Pressure (HBP) on April 1st, 2022 for Neighbourhoods in the City of Toronto
df56_merge = df56.merge(
    linkage_table,
    left_on = "Neighbourhood_df56",
    right_on = "HOOD_158",
    how = "left"
).rename(columns={"HOOD_140":"HOOD_140_df56"})
df56_final = df56_merge.groupby("HOOD_140_df56")["Pravelence_HighBloodPressure_df56"].mean().reset_index()

In [ ]:
# Adult Health and Disease: Prevalence (/100) of Mental Health and Addiction-related Visits to a Primary Care Physician 
# Jan-1st to Dec-31st, 2023 for Neighbourhoods in the City of Toronto
df57_merge = df57.merge(
    linkage_table,
    left_on = "Neighbourhood_df57",
    right_on = "HOOD_158",
    how = "left"
).rename(columns={"HOOD_140":"HOOD_140_df57"})
df57_final = df57_merge.groupby("HOOD_140_df57")["Pravelence_MentalHealthVisits_df57"].mean().reset_index()

In [ ]:
# Hospital Admissions: Rate (/1,000) of All Hospitalizations between April 1st, 2021 and March 31st, 2023 for Neighbourhoods in the City of Toronto 
df58_merge = df58.merge(
    linkage_table,
    left_on = "Neighbourhood_df58",
    right_on = "HOOD_158",
    how = "left"
).rename(columns={"HOOD_140":"HOOD_140_df58"})
df58_final = df58_merge.groupby("HOOD_140_df58")["AnnualRate_Hospitalizations_df58"].mean().reset_index()

In [ ]:
# "Mothers and Babies: Average Annual Hospital Live Birth Rate (/1,000) from Jan-1st, 2020 to 
# Dec-31st, 2022 in Toronto Neighbourhoods and Community Coordination Plan (CCP) Clusters
df59_merge = df59.merge(
    linkage_table,
    left_on = "Neighbourhood_df59",
    right_on = "HOOD_158",
    how = "left"
).rename(columns={"HOOD_140":"HOOD_140_df59"})
df59_final = df59_merge.groupby("HOOD_140_df59")["Birthrate_df59"].mean().reset_index()

In [ ]:
# Mothers and Babies: Percentage of Births to Women based on Mother's Country of Birth (MCOB) from Jan-1st, 2019 to 
# Dec-31st, 2021 in Toronto Neighbourhoods and Community Coordination Plan (CCP) Clusters
df60_merge = df60.merge(
    linkage_table,
    left_on = "Neighbourhood_df60",
    right_on = "HOOD_158",
    how = "left"
).rename(columns={"HOOD_140":"HOOD_140_df60"})
df60_final = df60_merge.groupby("HOOD_140_df60")["PercentBirths_MothersNotBornInCanada_df60"].mean().reset_index()

In [ ]:
# Prevalence (/100) of Asthma (2014/15) in Children and Youth on April 1st, 2015 ª in Toronto Neighbourhoods, Ages 0-24; 
df61_merge = df61.merge(
    linkage_table,
    left_on = "Neighbourhood_df61",
    right_on = "HOOD_158",
    how = "left"
).rename(columns={"HOOD_140":"HOOD_140_df61"})
df61_final = df61_merge.groupby("HOOD_140_df61")["Pravalence_AsthmaYouth_df61"].mean().reset_index()

In [ ]:
# Children and Youth: Average Annual Rate (/100) of Injuries for 2014/15 to 2015/16 
# (All Admissions and Emergency Department (ED) visits) in Toronto Neighbourhoods
df62_merge = df62.merge(
    linkage_table,
    left_on = "Neighbourhood_df62",
    right_on = "HOOD_158",
    how = "left"
).rename(columns={"HOOD_140":"HOOD_140_df62"})
df62_final = df62_merge.groupby("HOOD_140_df62")["Annual_Injury_ED_VisitsYouth_df62"].mean().reset_index()

In [ ]:
# Average annual rate (/1000) of Emergency Department (ED) visits that are Mental Health and 
# Addiction-related visits (among young adults aged 16-25) 2012/13 to 2014/15 
df63_merge = df63.merge(
    linkage_table,
    left_on = "Neighbourhood_df63",
    right_on = "HOOD_158",
    how = "left"
).rename(columns={"HOOD_140":"HOOD_140_df63"})
df63_final = df63_merge.groupby("HOOD_140_df63")["Annual_ED_MentalHealthVisits_df63"].mean().reset_index()

In [ ]:
# Adult Health and Disease: Prevalence (/100) of Congestive Heart Failure (CHF) Cases on April 1st, 2023ª for Neighbourhoods in the City of Toronto 
df64_merge = df64.merge(
    linkage_table,
    left_on = "Neighbourhood_df64",
    right_on = "HOOD_158",
    how = "left"
).rename(columns={"HOOD_140":"HOOD_140_df64"})
df64_final = df64_merge.groupby("HOOD_140_df64")["Pravalence_CongestiveHeartFailure_df64"].mean().reset_index()

In [ ]:
# Emergency Department (ED) Visits: Average Annual Rate (/1,000) of All ED Visits between April 1, 2023 
# and March 31, 2025 for Neighbourhoods in the City of Toronto
df65_merge = df65.merge(
    linkage_table,
    left_on = "Neighbourhood_df65",
    right_on = "HOOD_158",
    how = "left"
).rename(columns={"HOOD_140":"HOOD_140_df65"})
df65_final = df65_merge.groupby("HOOD_140_df65")["Annual_ED_Visits_df65"].mean().reset_index()

In [ ]:
# Emergency Department (ED) Visits: Average Annual Rate (/1,000) of Mental Health and Addiction-related ED Visits 
# between April 1, 2023 and March 31, 2025 for Neighbourhoods in the City of Toronto
df66_merge = df66.merge(
    linkage_table,
    left_on = "Neighbourhood_df66",
    right_on = "HOOD_158",
    how = "left"
).rename(columns={"HOOD_140":"HOOD_140_df66"})
df66_final = df66_merge.groupby("HOOD_140_df66")["Annual_MentalHealth_ED_df66"].mean().reset_index()

In [ ]:
# Hospital Admissions: Average Annual Rate (/1,000) of Mental Health and Addiction-related Hospitalizations between 
# April 1, 2021 and March 31, 2023ª for Neighbourhoods in the City of Toronto 
df67_merge = df67.merge(
    linkage_table,
    left_on = "Neighbourhood_df67",
    right_on = "HOOD_158",
    how = "left"
).rename(columns={"HOOD_140":"HOOD_140_df67"})
df67_final = df67_merge.groupby("HOOD_140_df67")["Annual_MentalHealth_Hospitalizations_df67"].mean().reset_index()

In [ ]:
# Average Annual Rate (/100,000) of Premature Mortality between 
# January 1, 2021 and December 31, 2023 for Neighbourhoods in the City of Toronto 
df68_merge = df68.merge(
    linkage_table,
    left_on = "Neighbourhood_df68",
    right_on = "HOOD_158",
    how = "left"
).rename(columns={"HOOD_140":"HOOD_140_df68"})
df68_final = df68_merge.groupby("HOOD_140_df68")["Annual_PrematureDeaths_df68"].mean().reset_index()

In [ ]:
# Proxy for sexual transmitted diseases
# Sexual Health: 5-Year Combined Rate (/100,000) of Chlamydia Cases
df69_merge = df69.merge(
    linkage_table,
    left_on = "Neighbourhood_df69",
    right_on = "HOOD_158",
    how = "left"
).rename(columns={"HOOD_140":"HOOD_140_df69"})
df69_final = df69_merge.groupby("HOOD_140_df69")["AvgChlaymdiaRate_df69"].mean().reset_index()

In [ ]:
# This dataset stores coordinates as latitude, longitude,
# so flip them when creating the Shapely point
def extract_point(value):

    if isinstance(value, str):
        value = json.loads(value)

    lat, lon = value["coordinates"][0]

    return Point(lon, lat)


# Convert geometry
df_geo = df70.copy()

df_geo["geometry_df70"] = (
    df_geo["geometry_df70"]
    .apply(extract_point)
)


# Convert to GeoDataFrame
df_geo = gpd.GeoDataFrame(
    df_geo,
    geometry="geometry_df70",
    crs="EPSG:4326"
)


# Match neighbourhood CRS
df_geo = df_geo.to_crs(
    neighbourhoods.crs
)


# Assign each location to neighbourhood 140
df_neighbourhood = gpd.sjoin(
    df_geo,
    neighbourhoods[
        [
            "AREA_SHORT_CODE",
            "geometry"
        ]
    ],
    how="left",
    predicate="within"
)


# Rename neighbourhood ID
df_neighbourhood = df_neighbourhood.rename(
    columns={
        "AREA_SHORT_CODE": "HOOD_140_df70"
    }
)


# Sum affordable housing units by neighbourhood
df70_counts = (
    df_neighbourhood
    .groupby(
        "HOOD_140_df70",
        as_index=False
    )["SumAffordableHousing_df70"]
    .sum()
)


# Get complete list of all 140 neighbourhoods
hood140 = (
    neighbourhoods[
        ["AREA_SHORT_CODE"]
    ]
    .drop_duplicates()
    .rename(
        columns={
            "AREA_SHORT_CODE": "HOOD_140_df70"
        }
    )
)


# Join housing values onto all 140 neighbourhoods
df70_final = hood140.merge(
    df70_counts,
    on="HOOD_140_df70",
    how="left"
)


# Neighbourhoods with no affordable housing observations = 0
df70_final["SumAffordableHousing_df70"] = (
    df70_final["SumAffordableHousing_df70"]
    .fillna(0)
)

In [ ]:
# Create counting function that groups by neighbourhood
def count_by_neighbourhood(
    df,
    neighbourhoods,
    linkage_table,
    geometry_col,
    df_num,
    count_col
):

    df_geo = df.copy()

    hood_col = f"HOOD_140_df{df_num}"

    # Convert geometry column

    def convert_geometry(value):

        if pd.isna(value):
            return None

        if isinstance(value, str):

            try:
                value = json.loads(value)

            except json.JSONDecodeError:
                value = ast.literal_eval(value)

        return shape(value)


    df_geo[geometry_col] = (
        df_geo[geometry_col]
        .apply(convert_geometry)
    )


    # Convert to GeoDataFrame

    df_geo = gpd.GeoDataFrame(
        df_geo,
        geometry=geometry_col,
        crs="EPSG:4326"
    )

    # Match neighbourhood CRS
    df_geo = df_geo.to_crs(
        neighbourhoods.crs
    )


    # Spatial join

    df_joined = gpd.sjoin(
        df_geo,
        neighbourhoods[
            [
                "AREA_SHORT_CODE",
                "geometry"
            ]
        ],
        how="left",
        predicate="within"
    )

    df_joined = df_joined.rename(
        columns={
            "AREA_SHORT_CODE": hood_col
        }
    )


    # Count observations per neighbourhood

    neighbourhood_counts = (
        df_joined
        .groupby(
            hood_col,
            as_index=False
        )
        .size()
        .rename(
            columns={
                "size": count_col
            }
        )
    )


    # Ensure all 140 neighbourhoods are included

    hood140 = (
        linkage_table[
            ["HOOD_140"]
        ]
        .drop_duplicates()
        .rename(
            columns={
                "HOOD_140": hood_col
            }
        )
    )


    neighbourhood_counts = hood140.merge(
        neighbourhood_counts,
        on=hood_col,
        how="left"
    )

    # Neighbourhoods with no observations = 0

    neighbourhood_counts[count_col] = (
        neighbourhood_counts[count_col]
        .fillna(0)
    )


    return neighbourhood_counts

In [ ]:
# Bridges
df71_final = count_by_neighbourhood(
    df=df71, 
    neighbourhoods=neighbourhoods, 
    linkage_table=linkage_table,
    geometry_col="geometry_df71", 
    df_num=71,
    count_col="NumberofBridges_df71"
)

In [ ]:
# CafeTO
df72_final = count_by_neighbourhood(
    df=df72, 
    neighbourhoods=neighbourhoods, 
    linkage_table=linkage_table,
    geometry_col="geometry_df72", 
    df_num=72,
    count_col="NumberofCafes_df72"
)

In [ ]:
# Clothing Dropbox
df73_final = count_by_neighbourhood(
    df=df73, 
    neighbourhoods=neighbourhoods, 
    linkage_table=linkage_table,
    geometry_col="geometry_df73", 
    df_num=73,
    count_col="NumberofClothingDropbox_df73"
)

In [ ]:
# Drinking fountains
df74_final = count_by_neighbourhood(
    df=df74, 
    neighbourhoods=neighbourhoods, 
    linkage_table=linkage_table,
    geometry_col="geometry_df74", 
    df_num=74,
    count_col="NumberofParkDrinkingFountains_df74"
)

In [ ]:
# Fire Hydrants
df75_final = count_by_neighbourhood(
    df=df75, 
    neighbourhoods=neighbourhoods, 
    linkage_table=linkage_table,
    geometry_col="geometry_df75", 
    df_num=75,
    count_col="NumberofFireHydrants_df75"
)

In [ ]:
# Food inspections outcomes

# Convert longitude/latitude columns to point geometry
# Already has all 140 neighbourhoods
df_geo = gpd.GeoDataFrame(
    df76.copy(),
    geometry=gpd.points_from_xy(
        df76["longitude_df76"],
        df76["latitude_df76"]
    ),
    crs="EPSG:4326"
)

# Reproject to match neighbourhood polygons
df_geo = df_geo.to_crs(
    neighbourhoods.crs
)

# Assign each point to a neighbourhood
df_joined = gpd.sjoin(
    df_geo,
    neighbourhoods[
        [
            "AREA_SHORT_CODE",
            "AREA_NAME",
            "geometry"
        ]
    ],
    how="left",
    predicate="within"
)

# Rename neighbourhood variables
df_joined = df_joined.rename(
    columns={
        "AREA_SHORT_CODE": "HOOD_140_df76",
        "AREA_NAME": "NEIGHBOURHOOD"
    }
)

df76_final = df_joined.groupby("HOOD_140_df76")["severity_ordinal_df76"].mean().reset_index().rename(columns={"severity_ordinal_df76":"FoodInspectionOutcome_df76"})

In [ ]:
# Air conditioned and cool spaces
df77_final = count_by_neighbourhood(
    df=df77, 
    neighbourhoods=neighbourhoods, 
    linkage_table=linkage_table,
    geometry_col="geometry_df77", 
    df_num=77,
    count_col="NumberofAirconditionedSpaces_df77"
)

In [ ]:
# Motor Vehicle Collisions
df78_final = count_by_neighbourhood(
    df=df78, 
    neighbourhoods=neighbourhoods, 
    linkage_table=linkage_table,
    geometry_col="geometry_df78", 
    df_num=78,
    count_col="NumberofMotorVehicleCollisions_df78"
)

In [ ]:
# Washrooms
df79_final = count_by_neighbourhood(
    df=df79, 
    neighbourhoods=neighbourhoods, 
    linkage_table=linkage_table,
    geometry_col="geometry_df79", 
    df_num=79,
    count_col="NumberofWashrooms_df79"
)

In [ ]:
# Parks and Rec
df80_final = count_by_neighbourhood(
    df=df80, 
    neighbourhoods=neighbourhoods, 
    linkage_table=linkage_table,
    geometry_col="geometry_df80", 
    df_num=80,
    count_col="NumberofParksandRec_df80"
)

In [ ]:
# Places of worship
df81_final = count_by_neighbourhood(
    df=df81, 
    neighbourhoods=neighbourhoods, 
    linkage_table=linkage_table,
    geometry_col="geometry_df81", 
    df_num=81,
    count_col="NumberofPlacesofWorship_df81"
)

In [ ]:
# PSS Licenses 
df82_final = count_by_neighbourhood(
    df=df82, 
    neighbourhoods=neighbourhoods, 
    linkage_table=linkage_table,
    geometry_col="geometry_df82", 
    df_num=82,
    count_col="NumberofPSSLicenses_df82"
)

In [ ]:
# Public art 
df83_final = count_by_neighbourhood(
    df=df83, 
    neighbourhoods=neighbourhoods, 
    linkage_table=linkage_table,
    geometry_col="geometry_df83", 
    df_num=83,
    count_col="NumberofArt_df83"
)

In [ ]:
# Real estate assets 
df84_final = count_by_neighbourhood(
    df=df84, 
    neighbourhoods=neighbourhoods, 
    linkage_table=linkage_table,
    geometry_col="geometry_df84", 
    df_num=84,
    count_col="NumberofRealEstateAssets_df84"
)

In [ ]:
# Land assets 
df85_final = count_by_neighbourhood(
    df=df85, 
    neighbourhoods=neighbourhoods, 
    linkage_table=linkage_table,
    geometry_col="geometry_df85", 
    df_num=85,
    count_col="NumberofLandAssets_df85"
)

In [ ]:
# Watch your speed program 
df86_final = count_by_neighbourhood(
    df=df86, 
    neighbourhoods=neighbourhoods, 
    linkage_table=linkage_table,
    geometry_col="geometry_df86", 
    df_num=86,
    count_col="NumberofSpeedDisplaySigns_df86"
)

In [ ]:
# School locations 
df88_final = count_by_neighbourhood(
    df=df88, 
    neighbourhoods=neighbourhoods, 
    linkage_table=linkage_table,
    geometry_col="geometry_df88", 
    df_num=88,
    count_col="NumberofSchools_df88"
)

In [ ]:
# Number of benches 
df89_final = count_by_neighbourhood(
    df=df89, 
    neighbourhoods=neighbourhoods, 
    linkage_table=linkage_table,
    geometry_col="geometry_df89", 
    df_num=89,
    count_col="NumberofBenches_df89"
)

In [ ]:
# Tennis courts 
df90_final = count_by_neighbourhood(
    df=df90, 
    neighbourhoods=neighbourhoods, 
    linkage_table=linkage_table,
    geometry_col="geometry_df90", 
    df_num=90,
    count_col="NumberofTennisCourts_df90"
)

In [ ]:
# Commercial Properties for sale or lease 
df91_final = count_by_neighbourhood(
    df=df91, 
    neighbourhoods=neighbourhoods, 
    linkage_table=linkage_table,
    geometry_col="geometry_df91", 
    df_num=91,
    count_col="NumberofCommericalIndustrialPropertiesforSale_df91"
)

In [ ]:
# Number of vehicles

# Convert longitude/latitude columns to point geometry
df_geo = gpd.GeoDataFrame(
    df92.copy(),
    geometry=gpd.points_from_xy(
        df92["longitude_df92"],
        df92["latitude_df92"]
    ),
    crs="EPSG:4326"
)

# Reproject to match neighbourhood polygons
df_geo = df_geo.to_crs(
    neighbourhoods.crs
)

# Assign each point to a neighbourhood
df_joined = gpd.sjoin(
    df_geo,
    neighbourhoods[
        [
            "AREA_SHORT_CODE",
            "AREA_NAME",
            "geometry"
        ]
    ],
    how="left",
    predicate="within"
)

# Rename neighbourhood variables
df_joined = df_joined.rename(
    columns={
        "AREA_SHORT_CODE": "HOOD_140_df92",
        "AREA_NAME": "NEIGHBOURHOOD"
    }
)

df92_final = df_joined.groupby("HOOD_140_df92")[["total_vehicle_df92","VehicleRatio_df92"]].mean().reset_index()

In [ ]:
# Number of Trees 
df93_final = count_by_neighbourhood(
    df=df93, 
    neighbourhoods=neighbourhoods, 
    linkage_table=linkage_table,
    geometry_col="geometry_df93", 
    df_num=93,
    count_col="NumberofTrees_df93"
)

In [ ]:
dfs = [
    (df1_final, "HOOD_140_df1"),
    (df2_final, "HOOD_140_df2"),
    (df3_final, "HOOD_140_df3"),
    (df4_final, "HOOD_140_df4"),
    (df5_final, "HOOD_140_df5"),
    (df12_final, "HOOD_140_df12"),
    (df13_final, "HOOD_140_df13"),
    (df15_final, "HOOD_140_df15"),
    (df16_final, "HOOD_140_df16"),
    (df23_final, "HOOD_140_df23"),
    (df24_final, "HOOD_140_df24"),
    (df25_final, "HOOD_140_df25"),
    (df26_final, "HOOD_140_df26"),
    (df27_final, "HOOD_140_df27"),
    (df32_final, "HOOD_140_df32"),
    (df35_final, "HOOD_140_df35"),
    (df36_final, "HOOD_140_df36"),
    (df37_final, "HOOD_140_df37"),
    (df38_final, "HOOD_140_df38"),
    (df39_final, "HOOD_140_df39"),
    (df40_final, "HOOD_140_df40"),
    (df41_final, "HOOD_140_df41"),
    (df45_final, "HOOD_140_df45"),
    (df46_final, "HOOD_140_df46"),
    (df47_final, "HOOD_140_df47"),
    (df48_final, "HOOD_140_df48"),
    (df49_final, "HOOD_140_df49"),
    (df51_final, "HOOD_140_df51"),
    (df52_final, "HOOD_140_df52"),
    (df53_final, "HOOD_140_df53"),
    (df54_final, "HOOD_140_df54"),
    (df55_final, "HOOD_140_df55"),
    (df56_final, "HOOD_140_df56"),
    (df57_final, "HOOD_140_df57"),
    (df58_final, "HOOD_140_df58"),
    (df59_final, "HOOD_140_df59"),
    (df60_final, "HOOD_140_df60"),
    (df61_final, "HOOD_140_df61"),
    (df62_final, "HOOD_140_df62"),
    (df63_final, "HOOD_140_df63"),
    (df64_final, "HOOD_140_df64"),
    (df65_final, "HOOD_140_df65"),
    (df66_final, "HOOD_140_df66"),
    (df67_final, "HOOD_140_df67"),
    (df68_final, "HOOD_140_df68"),
    (df69_final, "HOOD_140_df69"),
    (df70_final, "HOOD_140_df70"),
    (df71_final, "HOOD_140_df71"),
    (df72_final, "HOOD_140_df72"),
    (df73_final, "HOOD_140_df73"),
    (df74_final, "HOOD_140_df74"),
    (df75_final, "HOOD_140_df75"),
    (df76_final, "HOOD_140_df76"),
    (df77_final, "HOOD_140_df77"),
    (df78_final, "HOOD_140_df78"),
    (df79_final, "HOOD_140_df79"),
    (df80_final, "HOOD_140_df80"),
    (df81_final, "HOOD_140_df81"),
    (df82_final, "HOOD_140_df82"),
    (df83_final, "HOOD_140_df83"),
    (df84_final, "HOOD_140_df84"),
    (df85_final, "HOOD_140_df85"),
    (df86_final, "HOOD_140_df86"),
    (df88_final, "HOOD_140_df88"),
    (df89_final, "HOOD_140_df89"),
    (df90_final, "HOOD_140_df90"),
    (df91_final, "HOOD_140_df91"),
    (df92_final, "HOOD_140_df92"),
    (df93_final, "HOOD_140_df93")
]

df_merge_all = voters_neighbourhood_final.copy()

for df, key in dfs:

    df_merge_all = df_merge_all.merge(
        df,
        left_on="NEIGHBOURHOOD_ID",
        right_on=key,
        how="left"
    )

    # Remove the redundant HOOD_140_dfX join column
    df_merge_all = df_merge_all.drop(
        columns=key
    )

# Get rates of some variables based on population

In [ ]:
# Create variables with rate, identifying features in 100 people
df_merge_all["OFFENCE_RATE_df1"] = (df_merge_all["OFFENCE_df1"]/df_merge_all["Total Population_df32"])*100 
df_merge_all["CulturalHotspotsRate_df2"] = (df_merge_all["CulturalHotspots_df2"]/df_merge_all["Total Population_df32"])*100 
df_merge_all["HateCrimeRate_df3"] = (df_merge_all["HateCrimes_df3"]/df_merge_all["Total Population_df32"])*100 
df_merge_all["HomicideRate_df4"] = (df_merge_all["Homicides_df4"]/df_merge_all["Total Population_df32"])*100 
df_merge_all["MentalHealthApprehensionRate_df25"] = (df_merge_all["MentalHealthApprehensions_df25"]/df_merge_all["Total Population_df32"])*100 
df_merge_all["PoliceCallsAttendedRate_df36"] = (df_merge_all["PoliceCallsAttended_df36"]/df_merge_all["Total Population_df32"])*100 
df_merge_all["PoliceTicketRate_df37"] = (df_merge_all["TICKET_COUNT_df37"]/df_merge_all["Total Population_df32"])*100 
df_merge_all["ShootingsFirearmDisargeRate_df38"] = (df_merge_all["FirearmDisarges_df38"]/df_merge_all["Total Population_df32"])*100 
df_merge_all["SocialHousingUnitRate_df39"] = (df_merge_all["SocialHousingUnits_df39"]/df_merge_all["Total Population_df32"])*100
df_merge_all["TheftFromMotorVehicleRate_df40"] = (df_merge_all["TheftFromMotorVehicle_df40"]/df_merge_all["Total Population_df32"])*100 
df_merge_all["City Grants Funding $Rate_df41"] = (df_merge_all["City Grants Funding $_df41"]/df_merge_all["Total Population_df32"]) # per person instead of per 100
df_merge_all["Salvation Army DonorsRate_df41"] = (df_merge_all["Salvation Army Donors_df41"]/df_merge_all["Total Population_df32"])*100 
df_merge_all["Watermain BreaksRate_df41"] = (df_merge_all["Watermain Breaks_df41"]/df_merge_all["Total Population_df32"])*100 
df_merge_all["BusinessesRate_df45"] = (df_merge_all["Businesses_df45"]/df_merge_all["Total Population_df32"])*100 
df_merge_all["Child Care SpaceRate_df45"] = (df_merge_all["Child Care Spaces_df45"]/df_merge_all["Total Population_df32"])*100 
df_merge_all["EmployedRate_df45"] = (df_merge_all["Local Employment_df45"]/df_merge_all["Total Population_df32"])*100 
df_merge_all["Social Assistance RecipientsRate_df45"] = (df_merge_all["Social Assistance Recipients_df45"]/df_merge_all["Total Population_df32"])*100 
df_merge_all["Health ProvidersRate_df48"] = (df_merge_all["Health Providers_df48"]/df_merge_all["Total Population_df32"])*100 
df_merge_all["Student NutritionRate_df48"] = (df_merge_all["Student Nutrition_df48"]/df_merge_all["Total Population_df32"])*100 

# per person instead of per 100 people
df_merge_all["Green Rebate ProgramsRate_df47"] = (df_merge_all["Green Rebate Programs_df47"]/df_merge_all["Total Population_df32"])
df_merge_all["Green Spaces_df47"] = (df_merge_all["Green Spaces_df47"]/df_merge_all["Total Population_df32"])
df_merge_all["Tree Cover_df47"] = (df_merge_all["Tree Cover_df47"]/df_merge_all["Total Population_df32"])
df_merge_all["CrucialSeverityDineSafeInspectionsRate_df48"] = (df_merge_all["DineSafe Inspections_df48"]/df_merge_all["Total Population_df32"])
df_merge_all["NumberofBridgesRate_df71"] = (df_merge_all["NumberofBridges_df71"]/df_merge_all["Total Population_df32"])
df_merge_all["NumberofCafesRate_df72"] = (df_merge_all["NumberofCafes_df72"]/df_merge_all["Total Population_df32"])
df_merge_all["NumberofClothingDropboxRate_df73"] = (df_merge_all["NumberofClothingDropbox_df73"]/df_merge_all["Total Population_df32"])
df_merge_all["NumberofParkDrinkingFountainsRate_df74"] = (df_merge_all["NumberofParkDrinkingFountains_df74"]/df_merge_all["Total Population_df32"])
df_merge_all["NumberofFireHydrantsRate_df75"] = (df_merge_all["NumberofFireHydrants_df75"]/df_merge_all["Total Population_df32"])
df_merge_all["NumberofAirconditionedSpacesRate_df77"] = (df_merge_all["NumberofAirconditionedSpaces_df77"]/df_merge_all["Total Population_df32"])
df_merge_all["NumberofWashroomsRate_df79"] = (df_merge_all["NumberofWashrooms_df79"]/df_merge_all["Total Population_df32"])
df_merge_all["NumberofParksandRecRate_df80"] = (df_merge_all["NumberofParksandRec_df80"]/df_merge_all["Total Population_df32"])
df_merge_all["NumberofPlacesofWorshipRate_df81"] = (df_merge_all["NumberofPlacesofWorship_df81"]/df_merge_all["Total Population_df32"])
df_merge_all["NumberofPSSLicensesRate_df82"] = (df_merge_all["NumberofPSSLicenses_df82"]/df_merge_all["Total Population_df32"])
df_merge_all["NumberofArtRate_df83"] = (df_merge_all["NumberofArt_df83"]/df_merge_all["Total Population_df32"])
df_merge_all["NumberofRealEstateAssetsRate_df84"] = (df_merge_all["NumberofRealEstateAssets_df84"]/df_merge_all["Total Population_df32"])
df_merge_all["NumberofLandAssetsRate_df85"] = (df_merge_all["NumberofLandAssets_df85"]/df_merge_all["Total Population_df32"])
df_merge_all["NumberofSpeedDisplaySignsRate_df86"] = (df_merge_all["NumberofSpeedDisplaySigns_df86"]/df_merge_all["Total Population_df32"])
df_merge_all["NumberofSchoolsRate_df88"] = (df_merge_all["NumberofSchools_df88"]/df_merge_all["Total Population_df32"])
df_merge_all["NumberofBenchesRate_df89"] = (df_merge_all["NumberofBenches_df89"]/df_merge_all["Total Population_df32"])
df_merge_all["NumberofTennisCourtsRate_df90"] = (df_merge_all["NumberofTennisCourts_df90"]/df_merge_all["Total Population_df32"])
df_merge_all["NumberofCommericalIndustrialPropertiesforSaleRate_df91"] = (df_merge_all["NumberofCommericalIndustrialPropertiesforSale_df91"]/df_merge_all["Total Population_df32"])

# back to per 100
df_merge_all["Rent Bank ApplicantsRate_df49"] = (df_merge_all["Rent Bank Applicants_df49"]/df_merge_all["Total Population_df32"])*100 
df_merge_all["SumAffordableHousingRate_df70"] = (df_merge_all["SumAffordableHousing_df70"]/df_merge_all["Total Population_df32"])*100 
df_merge_all["NumberofMotorVehicleCollisionsRate_df78"] = (df_merge_all["NumberofMotorVehicleCollisions_df78"]/df_merge_all["Total Population_df32"])*100 
df_merge_all["total_vehicleRate_df92"] = (df_merge_all["total_vehicle_df92"]/df_merge_all["Total Population_df32"])*100 
df_merge_all["NumberofTreesRate_df93"] = (df_merge_all["NumberofTrees_df93"]/df_merge_all["Total Population_df32"])*100 

df_final = df_merge_all.drop(columns=[
    "OFFENCE_df1",
    "CulturalHotspots_df2",
    "HateCrimes_df3",
    "Homicides_df4",
    "MentalHealthApprehensions_df25",
    "PoliceCallsAttended_df36",
    "TICKET_COUNT_df37",
    "FirearmDisarges_df38",
    "SocialHousingUnits_df39",
    "TheftFromMotorVehicle_df40",
    "City Grants Funding $_df41",
    "Salvation Army Donors_df41",
    "Watermain Breaks_df41",
    "Businesses_df45",
    "Child Care Spaces_df45",
    "Local Employment_df45",
    "Social Assistance Recipients_df45",
    "Green Rebate Programs_df47",
    "Green Spaces_df47",
    "Tree Cover_df47",
    "DineSafe Inspections_df48",
    "Health Providers_df48",
    "Student Nutrition_df48",
    "Rent Bank Applicants_df49",
    "SumAffordableHousing_df70",
    "NumberofBridges_df71",
    "NumberofCafes_df72",
    "NumberofClothingDropbox_df73",
    "NumberofParkDrinkingFountains_df74",
    "NumberofFireHydrants_df75",
    "NumberofAirconditionedSpaces_df77",
    "NumberofMotorVehicleCollisions_df78",
    "NumberofWashrooms_df79",
    "NumberofParksandRec_df80",
    "NumberofPlacesofWorship_df81",
    "NumberofPSSLicenses_df82",
    "NumberofArt_df83",
    "NumberofRealEstateAssets_df84",
    "NumberofLandAssets_df85",
    "NumberofSpeedDisplaySigns_df86",
    "NumberofSchools_df88",
    "NumberofBenches_df89",
    "NumberofTennisCourts_df90",
    "NumberofCommericalIndustrialPropertiesforSale_df91",
    "total_vehicle_df92",
    "NumberofTrees_df93"
])
df_final = df_final.reset_index(drop=True)

df_final.to_excel(
    "final_table.xlsx",
    index=False
)

C:\Users\owner\AppData\Local\Temp\ipykernel_20848\3820499061.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_merge_all["OFFENCE_RATE_df1"] = (df_merge_all["OFFENCE_df1"]/df_merge_all["Total Population_df32"])*100
C:\Users\owner\AppData\Local\Temp\ipykernel_20848\3820499061.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_merge_all["CulturalHotspotsRate_df2"] = (df_merge_all["CulturalHotspots_df2"]/df_merge_all["Total Population_df32"])*100
C:\Users\owner\AppData\Local\Temp\ipykernel_20848\3820499061.py:4: Performanc